# Extension: MC Dropout uncertainty and a volatility-regime scanner for MT

MT (and the paper's models generally) output a single probability per factor per month with
no sense of how much to trust that number, and no rule for when to sit out entirely. This
notebook builds two independent ways to add that missing risk signal — one that asks the
*model* how confident it is, the other that ignores the model and asks the *return series*
whether the environment is currently calm or turbulent — and compares four resulting
strategies against MT's unweighted baseline.

## Idea 1: MC Dropout — ask the model

Monte Carlo Dropout (Gal and Ghahramani, 2016) is a way to get an uncertainty estimate
essentially for free from a network that already uses dropout for regularization.

**Normally**, dropout randomly zeroes a fraction of neurons *during training only*, so the
network can't lean on any single narrow path through its own representation — it's switched
off at prediction time so the network's full capacity is used.

**MC Dropout's reframe**: leave dropout switched *on* at prediction time, and instead of asking
the network once, ask it the same question 30-100 times. Each pass silences a different random
subset of neurons, so each pass can give a slightly different answer — not because the world
changed, but because each pass samples a slightly different internal path through what the
network learned. If the input sits in territory the model understands well, the passes agree
closely. If it doesn't, the passes scatter. That scatter is the signal: it's not "what's the
answer," it's "how much do internally-varied versions of my own reasoning agree with each
other" — an empirical, after-the-fact check on whether the model has a stable, well-supported
answer or is essentially guessing behind a confident-looking number.

## Idea 2: volatility scanner — ask the return series

A simpler, model-free companion: instead of perturbing the network, look at each factor's own
trailing realized volatility (rolling 3-month std of its return) — no model, no look-ahead,
only what's already known by the signal date. Flag, and abstain on, a factor-month whenever
that trailing volatility exceeds a cutoff calibrated per fold from that fold's validation-period
data. This tests a different hypothesis from MC Dropout: not "does the model doubt itself" but
"is the recent environment turbulent" — and, per the comparison at the end of this notebook, it
turns out to catch a mostly disjoint set of risky months (see the README for the headline
result: of the two, the volatility scanner is the one that actually beats the baseline).

**What this notebook does:**
1. Train an MT variant with dropout layers in the shared trunk — same walk-forward procedure
   as `Initial MT.ipynb`, nothing else changes. Model + walk-forward logic live in
   `src/mcdropout.py`, not inline here, so a single diagnostic run and the repeated-run sweep
   below can't drift apart from each other.
2. At prediction time, run each month's predictors through the network 50 times with dropout
   forced on, per factor. The mean of those 50 passes is the point estimate (same role as MT's
   usual single output); the spread (std) is the MC-Dropout uncertainty signal.
3. Calibrate two cutoffs each fold, both from that fold's validation-period data only — never
   test, to avoid leakage: an MC-Dropout uncertainty cutoff (e.g. flag the most-uncertain 20% of
   factor-months), and a separate trailing-volatility cutoff for the scanner.
4. Compare four strategies against the unweighted baseline: two ways of acting on MC-Dropout
   uncertainty (shrink position size in proportion to uncertainty, or sit out flagged months
   entirely) and the volatility scanner's abstain rule — then check whether the scanner is just
   re-flagging the months MC Dropout already flags, or catching something genuinely different.
5. **Report the headline performance numbers as a range across 5 independent reruns, not a
   single number.** Same code, same fixed seed every time — but TensorFlow training isn't
   perfectly deterministic (documented in the main README), and for this model that spread is
   large enough to matter: the volatility scanner's t(alpha) has been observed anywhere from
   ~1.5 to ~2.7 across reruns with nothing changed. A single run can make (or break) the
   "beats baseline" claim by luck alone, so the number worth trusting is the multi-run summary
   at the bottom of this notebook, not any one run above it.

Architecture note: the paper's MT uses batch normalization, not dropout — this is a deliberate
addition for this extension. Forcing `training=True` to activate dropout at prediction time
would *also* put batch norm into batch-statistics mode (a well-known MC-Dropout pitfall), so
this variant's shared trunk uses dropout in place of batch norm instead of combining both.

In [1]:
import sys, os
sys.path.append(os.path.abspath('../src'))

import numpy as np
import pandas as pd

import loading
import estimation as est
import mcdropout as mcd

FACTOR_NAMES = est.FACTOR_NAMES

In [2]:
# Same walk-forward estimation procedure as Initial MT.ipynb (paper Sec 3.2.4) — this notebook
# only changes what happens at *prediction* time, not the training/validation/test split.

data, feature_cols = est.build_labels_and_panel(
    loading.response_factors, loading.macro_predictors, loading.financial_predictors
)

print("data shape:", data.shape)
print("features:", len(feature_cols))
print("date range:", data.index.min().date(), "to", data.index.max().date())

data shape: (683, 264)
features: 259
date range: 1965-01-31 to 2021-11-30


In [3]:
# Model architecture, MC-Dropout sampling, and walk-forward training all live in
# src/mcdropout.py (imported above as `mcd`) — including the rationale for l1=0.001 instead of
# the paper's grid, which is a longer story than fits as a notebook comment. Quick sanity check
# with fake data before committing to the real (much longer) run below.

n_features = len(feature_cols)
model = mcd.build_mt_mcdropout_model(n_features)
X_fake = np.random.randn(20, n_features).astype('float32')
mc_fake = mcd.mc_dropout_predict(model, X_fake, n_samples=10)
print("MC sample array shape (n_samples, n_rows, n_factors):", mc_fake.shape)
print("Model builds and MC-samples without error.")

MC sample array shape (n_samples, n_rows, n_factors): (10, 20, 5)
Model builds and MC-samples without error.


In [4]:
# Train + MC-Dropout predict across all OOS folds, N_STABILITY_RUNS times over (same code,
# same fixed seed=0 every time — see markdown above for why repeating this matters here).
# Each run independently builds all four strategies and their performance table; run_stability
# stacks them into `all_perf` and reduces to `summary` (min/mean/max per metric per strategy).
#
# The diagnostics in the next few cells (confidence-weight calibration, the four-worst-months
# check, MC-Dropout/vol-regime overlap) are all one-run-at-a-time questions about a specific set
# of trained models and predictions — they don't have a sensible "average across runs" version,
# so they use `runs[0]` (the first of the N_STABILITY_RUNS runs, nothing special about it) as a
# representative single run. The *performance numbers* reported at the very end of this notebook
# use the full `summary` across all runs, not run 0 alone.

N_STABILITY_RUNS = 5
runs, all_perf, summary = mcd.run_stability(data, feature_cols, loading.response_factors, n_runs=N_STABILITY_RUNS)
run0 = runs[0]

run0['oos_mean'].to_csv('../results/mcdropout_oos_predictions.csv')
run0['oos_std'].to_csv('../results/mcdropout_oos_uncertainty.csv')
run0['oos_flag'].to_csv('../results/mcdropout_oos_flag.csv')
run0['oos_conviction_cutoff'].to_csv('../results/mcdropout_oos_conviction_cutoff.csv')
run0['oos_vol_flag'].to_csv('../results/mcdropout_oos_vol_flag.csv')

print(f"{N_STABILITY_RUNS} full walk-forward runs complete. OOS predictions per run: {run0['oos_mean'].shape}")

5 full walk-forward runs complete. OOS predictions per run: (383, 5)


In [5]:
# Confidence-scaled strategy check (run 0): does the confidence weight actually come out lower
# on the months MC-Dropout flagged as uncertain? If not, the "shrink position with uncertainty"
# strategy is just adding noise to position size, not risk management.

print("mean confidence weight on flagged vs. unflagged factor-months (run 0):")
print(f"  flagged:   {run0['confidence_weight'].values[run0['flag_vals'].values].mean():.2f}")
print(f"  unflagged: {run0['confidence_weight'].values[~run0['flag_vals'].values].mean():.2f}")

mean confidence weight on flagged vs. unflagged factor-months (run 0):
  flagged:   0.26
  unflagged: 0.69


In [6]:
# --- Vol-regime abstain (run 0): an independent volatility-regime overlay, NOT another
# MC-Dropout variant. A prior diagnostic found the worst baseline losses (Feb 2000 SMB, Dec
# 2008 HML, the 2009 momentum-crash months, May 2021 HML) happen in HIGH-conviction,
# LOW-uncertainty months -- MC Dropout's 50 stochastic passes all learned the same about-to-break
# pattern together, so nothing about the model's own self-assessment flags them. Conviction-gating
# was rejected for the same reason (it would keep exactly these high-conviction positions at full
# size).
#
# This strategy doesn't use MC Dropout uncertainty or conviction anywhere -- it abstains purely
# on trailing realized return volatility (rolling 3-month std, computed inside
# src/mcdropout.py's run_walkforward), a model-free, look-ahead-free feature of the return series
# itself, to test whether an independent regime signal catches what the model's own
# self-assessment missed. Cutoff calibrated per fold from that fold's *validation*-period
# trailing vol only (same UNCERTAINTY_QUANTILE discipline as everywhere else) -- never from test.
# Threshold/window were fixed in advance (3-month, 80th percentile) and not tuned against these
# results.

print(f"vol-regime abstain: flagged {run0['oos_vol_flag'].values.mean():.1%} of factor-months (run 0)")

vol-regime abstain: flagged 22.5% of factor-months (run 0)


In [7]:
# Is the MC-Dropout uncertainty signal actually informative (run 0)? If it's meaningful,
# flagged (high-uncertainty) factor-months should have *lower* classification accuracy than
# unflagged ones.
correct, flag_vals = run0['correct'], run0['flag_vals']
acc_flagged = correct.values[flag_vals.values].mean()
acc_unflagged = correct.values[~flag_vals.values].mean()
print(f"Accuracy on flagged (uncertain) factor-months:   {acc_flagged:.1%}  (n={flag_vals.values.sum()})")
print(f"Accuracy on unflagged (confident) factor-months: {acc_unflagged:.1%}  (n={(~flag_vals.values).sum()})")

# vol-regime abstain (run 0): did it actually catch the specific months that broke the baseline?
oos_vol_flag = run0['oos_vol_flag']
print("\nvol-regime abstain -- hit/miss on the four worst diagnosed losses:")
named_months = [('2000-02-29', 'SMB'), ('2008-12-31', 'HML'), ('2009-04-30', 'MOM'), ('2021-05-31', 'HML')]
for dt, f in named_months:
    hit = bool(oos_vol_flag.loc[dt, f])
    print(f"  {dt} {f}: {'ABSTAINED (hit)' if hit else 'still exposed (miss)'}")

# does trailing-vol flag genuinely different months than MC-Dropout uncertainty, or just the
# same ones by another name? (run 0)
vol_flagged_mask = oos_vol_flag.values
overlap_with_mc = flag_vals.values[vol_flagged_mask].mean()
print(f"\nOf all vol-regime-flagged factor-months (n={vol_flagged_mask.sum()}), "
      f"{overlap_with_mc:.1%} were also MC-Dropout-flagged as uncertain "
      f"({1 - overlap_with_mc:.1%} were months MC-Dropout was confident about).")

Accuracy on flagged (uncertain) factor-months:   50.9%  (n=464)
Accuracy on unflagged (confident) factor-months: 56.0%  (n=1451)

vol-regime abstain -- hit/miss on the four worst diagnosed losses:
  2000-02-29 SMB: ABSTAINED (hit)
  2008-12-31 HML: ABSTAINED (hit)
  2009-04-30 MOM: ABSTAINED (hit)
  2021-05-31 HML: ABSTAINED (hit)

Of all vol-regime-flagged factor-months (n=431), 32.7% were also MC-Dropout-flagged as uncertain (67.3% were months MC-Dropout was confident about).


## Headline result: performance across 5 independent reruns

Everything above uses `run0`, one single walk-forward pass, for diagnostics that are
inherently about one specific set of trained models (did *this* run catch *this* month, was
*this* run's uncertainty signal calibrated). But the performance numbers that actually support
the "beats baseline" claim — Sharpe, alpha, t(alpha), beta, R² — are exactly the numbers that
move most between runs, because TensorFlow training isn't perfectly deterministic even with a
fixed seed. This is not a hypothetical concern: rerunning this exact notebook has previously
produced a t(alpha) for the volatility scanner anywhere from ~1.5 to ~2.7.

So the table below is the min/mean/max of each metric across all `N_STABILITY_RUNS` reruns
computed above, not a single draw — this is the number worth citing, and it belongs at the
bottom of this notebook because it's the most current, most defensible one.

In [8]:
print(f"Multi-factor timing performance vs. multi-factor BUY, across {N_STABILITY_RUNS} independent reruns:")
print(summary.round(3).to_string())
print(f"\nmulti-factor BUY Sharpe (run 0, deterministic given fixed data): {est.annualized_sharpe(run0['buy_ew']):.2f}")

Multi-factor timing performance vs. multi-factor BUY, across 5 independent reruns:


                             baseline (no uncertainty)                 confidence-scaled                 abstain on flagged                 vol-regime abstain                
                                                   min    mean     max               min    mean     max                min    mean     max                min    mean     max
metric                                                                                                                                                                        
Sharpe Ratio                                     0.553   0.650   0.701             0.453   0.559   0.636              0.336   0.557   0.715              0.581   0.685   0.779
alpha (annualized %)                             0.065   0.513   0.835            -0.079   0.277   0.545             -0.515   0.411   1.079              0.766   1.089   1.345
t(alpha)                                         0.184   1.254   1.787            -0.205   0.715   1.289             -0.844  

In [9]:
# Synthesis, computed from the summary/all_perf tables above rather than hardcoded, so it stays
# accurate on any future rerun of this notebook.
strategies = ['baseline (no uncertainty)', 'confidence-scaled', 'abstain on flagged', 'vol-regime abstain']
mean_talpha = {s: summary.loc['t(alpha)', (s, 'mean')] for s in strategies}
mean_sharpe = {s: summary.loc['Sharpe Ratio', (s, 'mean')] for s in strategies}
best_talpha_strategy = max(mean_talpha, key=mean_talpha.get)
best_sharpe_strategy = max(mean_sharpe, key=mean_sharpe.get)

vol_lo, vol_mean, vol_hi = (summary.loc['t(alpha)', ('vol-regime abstain', s)] for s in ['min', 'mean', 'max'])
vol_talpha_per_run = all_perf.xs('t(alpha)', level='metric')['vol-regime abstain']
n_below_2 = (vol_talpha_per_run < 2.0).sum()
buy_sharpe = est.annualized_sharpe(run0['buy_ew'])

print(f"""Across {N_STABILITY_RUNS} reruns:

- Highest mean t(alpha): '{best_talpha_strategy}' ({mean_talpha[best_talpha_strategy]:.2f})
- Highest mean Sharpe:   '{best_sharpe_strategy}' ({mean_sharpe[best_sharpe_strategy]:.2f}, vs. BUY {buy_sharpe:.2f})
- vol-regime abstain t(alpha) range: [{vol_lo:.2f}, {vol_hi:.2f}], mean {vol_mean:.2f}
  -- {n_below_2}/{N_STABILITY_RUNS} reruns landed below conventional significance (t = 2),
  {N_STABILITY_RUNS - n_below_2}/{N_STABILITY_RUNS} above it.

Same conclusion as the single-run analysis this notebook used to report, now backed by a range
instead of one number: the volatility scanner is the strategy most consistently worth trusting,
MC-Dropout uncertainty (scaled or abstained-on) is not.""")

Across 5 reruns:

- Highest mean t(alpha): 'vol-regime abstain' (2.21)
- Highest mean Sharpe:   'vol-regime abstain' (0.69, vs. BUY 0.60)
- vol-regime abstain t(alpha) range: [1.53, 2.88], mean 2.21
  -- 2/5 reruns landed below conventional significance (t = 2),
  3/5 above it.

Same conclusion as the single-run analysis this notebook used to report, now backed by a range
instead of one number: the volatility scanner is the strategy most consistently worth trusting,
MC-Dropout uncertainty (scaled or abstained-on) is not.
